# 100K Row Scaling Benchmark (2xT4 Multi-GPU)

- **Subsample workflow**: 5K init -> pre-sweeps -> batch insert -> full 100K inference
- **Data connector roundtrip**: `save_npy` / `load_npy_mmap` at 100K scale
- **Multi-chain pmap inference**: chains distributed across GPUs via `jax.pmap`

Designed for Kaggle 2xT4 (32GB total VRAM).

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os

WORKDIR = "/kaggle/working/jaxcross"
BRANCH = "main"  # @param {type:"string"}

!git clone https://github.com/sambhal-labs/jaxcross.git {WORKDIR} 2>/dev/null \
    || (cd {WORKDIR} && git pull)
os.chdir(WORKDIR)

!git fetch origin && (git checkout {BRANCH} || git checkout -b {BRANCH} origin/{BRANCH}) && git pull origin {BRANCH}

%pip install -e . --no-deps -q

print(f"Branch: {BRANCH}")
print(f"Working directory: {os.getcwd()}")
print("Setup complete.")

In [ ]:
import gc
import json
import shutil
import tempfile
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np

import crosscat.packed.state as _ps
from benchmarks.utils import detect_platform, make_benchmark_data
from crosscat import (
    initialize,
    load_npy_mmap,
    pack_state,
    packed_gibbs_sweep,
    packed_insert_rows,
    save_npy,
    suggest_max_clusters,
)
from crosscat.packed import batch_packed_states
from crosscat.packed.kernels import packed_log_joint

platform = detect_platform()
print(f"Platform: {platform['platform']}, Backend: {platform['backend']}")
print(f"GPUs: {platform['n_gpus']}x {platform['gpu_names']}")

n_devices = jax.device_count()
print(f"JAX devices: {n_devices} ({jax.devices()})")
assert jax.default_backend() in ("gpu", "tpu"), "This notebook requires GPU/TPU runtime!"

## 2. Configuration

In [ ]:
N_ROWS = 100_000
N_COLS = 20
N_CHAINS = max(2, n_devices)  # At least 1 chain per GPU
CHAINS_PER_DEVICE = N_CHAINS // n_devices
N_CHAINS = CHAINS_PER_DEVICE * n_devices  # Round to divisible
assert CHAINS_PER_DEVICE >= 1, f"Need at least 1 chain per device: N_CHAINS must be >= n_devices ({n_devices})"
SUBSAMPLE_SIZE = 5000
INSERT_BATCH_SIZE = 5000
N_PRE_SWEEPS = 5
N_POST_SWEEPS = 10
DIAG_INTERVAL = 5
SEED = 42

print(f"Config: {N_ROWS:,} rows x {N_COLS} cols, {N_CHAINS} chains ({CHAINS_PER_DEVICE}/device)")
print(f"Subsample: {SUBSAMPLE_SIZE}, Post-sweeps: {N_POST_SWEEPS}")
print(f"suggest_max_clusters({N_ROWS}) = {suggest_max_clusters(N_ROWS)}")

## 3. Data Connector Roundtrip

Test `save_npy` / `load_npy_mmap` at 100K scale.

In [ ]:
print("--- Data Connector Roundtrip: 100,000 x 20 ---")
k_rt = jax.random.fold_in(jax.random.key(SEED), 1)
rt_data, rt_col_types = make_benchmark_data(k_rt, N_ROWS, N_COLS)
col_names = [f"col_{j}" for j in range(N_COLS)]

tmp_dir = Path(tempfile.mkdtemp(prefix="jaxcross_bench_"))
npy_path = tmp_dir / "test_100k.npy"

t0 = time.perf_counter()
save_npy(npy_path, rt_data, column_names=col_names)
save_time = time.perf_counter() - t0
file_size_mb = npy_path.stat().st_size / (1024 * 1024)
print(f"  save_npy: {save_time:.2f}s ({file_size_mb:.1f} MB)")

t0 = time.perf_counter()
loaded_data, loaded_names = load_npy_mmap(npy_path)
load_time = time.perf_counter() - t0
print(f"  load_npy_mmap: {load_time:.2f}s")

assert loaded_data.shape == rt_data.shape
assert np.allclose(loaded_data, np.asarray(rt_data), equal_nan=True)
assert loaded_names == col_names
print("  roundtrip verified OK")

# Cleanup
npy_path.unlink(missing_ok=True)
npy_path.with_suffix(".json").unlink(missing_ok=True)
shutil.rmtree(tmp_dir, ignore_errors=True)
del rt_data, loaded_data
gc.collect()

## 4. Define pmap Sweep Function

Distribute chains across GPUs using `jax.pmap`. Each GPU runs `CHAINS_PER_DEVICE` chains via `lax.fori_loop`.

In [ ]:
def _sweep_one_chain(key, packed, data, n_sweeps):
    """Run n_sweeps of packed Gibbs on a single chain."""
    return packed_gibbs_sweep(key, packed, data, n_sweeps=n_sweeps)


def _sweep_chains_on_device(keys, packed_batch, data, n_sweeps):
    """Run sweeps for CHAINS_PER_DEVICE chains on one device.

    keys: (chains_per_device,) array of PRNG keys
    packed_batch: batched PackedCrossCatState with leading (chains_per_device,) dim
    """

    def body(i, carry):
        packed_b = carry
        single_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)[i]
        for name in _ps._STATIC_FIELDS:
            single_kwargs[name] = getattr(packed_b, name)
        single = _ps.PackedCrossCatState(**single_kwargs)

        result = _sweep_one_chain(keys[i], single, data, n_sweeps)

        new_kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            arr = getattr(packed_b, name)
            new_kwargs[name] = arr.at[i].set(getattr(result, name))
        for name in _ps._STATIC_FIELDS:
            new_kwargs[name] = getattr(packed_b, name)
        return _ps.PackedCrossCatState(**new_kwargs)

    return jax.lax.fori_loop(0, keys.shape[0], body, packed_batch)


pmap_sweep = jax.pmap(
    _sweep_chains_on_device,
    in_axes=(0, 0, None, None),
    static_broadcasted_argnums=(3,),
)

print("pmap sweep function defined (will compile on first call)")

## 5. Multi-Chain Subsample Workflow (pmap)

Init on subsample -> pre-sweeps -> batch insert -> post-sweeps with diagnostics.
Chains are distributed across GPUs via pmap.

In [ ]:
key = jax.random.key(SEED)
k_data, k_init = jax.random.split(key)

# Generate data
data, col_types = make_benchmark_data(k_data, N_ROWS, N_COLS)
max_k = suggest_max_clusters(N_ROWS)
print(f"Data: {data.nbytes / (1024**2):.1f} MB, max_clusters: {max_k}")

# Initialize N_CHAINS on subsample
init_keys = jax.random.split(k_init, N_CHAINS)
all_packed = []
sub_idx = None
for c in range(N_CHAINS):
    result = initialize(init_keys[c], data, col_types, subsample_rows=SUBSAMPLE_SIZE)
    all_packed.append(pack_state(result.state, max_clusters=max_k))
    if sub_idx is None:
        sub_idx = result.subsample_idx
print(f"Initialized {N_CHAINS} chains on {SUBSAMPLE_SIZE} rows")

sub_data = data[sub_idx]

# Pre-sweeps on subsample (pmap)
print(f"\nPre-sweeps ({N_PRE_SWEEPS} sweeps on subsample)...")
sweep_keys = jax.random.split(jax.random.key(SEED + 100), N_CHAINS)
batched = batch_packed_states(all_packed)
keys_pmap = sweep_keys.reshape(n_devices, CHAINS_PER_DEVICE, *sweep_keys.shape[1:])
batched_pmap_kwargs = {}
for name in _ps._ARRAY_FIELDS:
    arr = getattr(batched, name)
    batched_pmap_kwargs[name] = arr.reshape((n_devices, CHAINS_PER_DEVICE) + arr.shape[1:])
for name in _ps._STATIC_FIELDS:
    batched_pmap_kwargs[name] = getattr(batched, name)
batched_pmap = _ps.PackedCrossCatState(**batched_pmap_kwargs)

t0 = time.perf_counter()
result_pmap = pmap_sweep(keys_pmap, batched_pmap, sub_data, N_PRE_SWEEPS)
jax.tree.map(lambda x: x.block_until_ready(), result_pmap)
pre_time = time.perf_counter() - t0
print(f"  Pre-sweeps: {pre_time:.2f}s (includes JIT compile)")

# Unflatten results
for c in range(N_CHAINS):
    dev_idx = c // CHAINS_PER_DEVICE
    chain_in_dev = c % CHAINS_PER_DEVICE
    kwargs = {}
    for name in _ps._ARRAY_FIELDS:
        kwargs[name] = getattr(result_pmap, name)[dev_idx][chain_in_dev]
    for name in _ps._STATIC_FIELDS:
        kwargs[name] = getattr(result_pmap, name)
    all_packed[c] = _ps.PackedCrossCatState(**kwargs)

# Batch insert remaining rows (sequential per chain -- same data insertion for all)
remaining_mask = jnp.ones(N_ROWS, dtype=bool).at[sub_idx].set(False)
remaining_idx = jnp.where(remaining_mask, size=N_ROWS - SUBSAMPLE_SIZE)[0]
remaining_data = data[remaining_idx]
n_batches = (remaining_data.shape[0] + INSERT_BATCH_SIZE - 1) // INSERT_BATCH_SIZE

print(f"\nInserting {remaining_data.shape[0]:,} rows in {n_batches} batches...")
t0 = time.perf_counter()
current_data = sub_data
for b in range(n_batches):
    batch = remaining_data[b * INSERT_BATCH_SIZE : (b + 1) * INSERT_BATCH_SIZE]
    for c in range(N_CHAINS):
        kb = jax.random.fold_in(jax.random.key(SEED + 200 + c), b)
        all_packed[c], _ = packed_insert_rows(kb, all_packed[c], current_data, batch)
    # Update current_data once (all chains share same data)
    current_data = jnp.concatenate([current_data, batch], axis=0)
    if (b + 1) % 5 == 0 or b == n_batches - 1:
        print(
            f"  Batch {b + 1}/{n_batches}: {all_packed[0].n_rows:,} rows, "
            f"{time.perf_counter() - t0:.1f}s"
        )
insert_time = time.perf_counter() - t0
print(f"Insert complete: {insert_time:.2f}s")
full_data = current_data

## 6. Post-Sweeps with Diagnostics (pmap)

Run Gibbs sweeps on full data, collecting diagnostics periodically.

In [ ]:
print(f"Post-sweeps: {N_POST_SWEEPS} sweeps on {all_packed[0].n_rows:,} rows")
sweep_key = jax.random.key(SEED + 300)
all_metrics = [[] for _ in range(N_CHAINS)]

sweep = 0
while sweep < N_POST_SWEEPS:
    batch = min(DIAG_INTERVAL, N_POST_SWEEPS - sweep)
    sweep_key, subkey = jax.random.split(sweep_key)
    chain_keys = jax.random.split(subkey, N_CHAINS)

    # Batch + reshape for pmap
    batched = batch_packed_states(all_packed)
    keys_pmap = chain_keys.reshape(n_devices, CHAINS_PER_DEVICE, *chain_keys.shape[1:])
    batched_pmap_kwargs = {}
    for name in _ps._ARRAY_FIELDS:
        arr = getattr(batched, name)
        batched_pmap_kwargs[name] = arr.reshape((n_devices, CHAINS_PER_DEVICE) + arr.shape[1:])
    for name in _ps._STATIC_FIELDS:
        batched_pmap_kwargs[name] = getattr(batched, name)
    batched_pmap = _ps.PackedCrossCatState(**batched_pmap_kwargs)

    t0 = time.perf_counter()
    result_pmap = pmap_sweep(keys_pmap, batched_pmap, full_data, batch)
    jax.tree.map(lambda x: x.block_until_ready(), result_pmap)
    elapsed = time.perf_counter() - t0
    sweep += batch

    # Unflatten + diagnostics
    for c in range(N_CHAINS):
        dev_idx = c // CHAINS_PER_DEVICE
        chain_in_dev = c % CHAINS_PER_DEVICE
        kwargs = {}
        for name in _ps._ARRAY_FIELDS:
            kwargs[name] = getattr(result_pmap, name)[dev_idx][chain_in_dev]
        for name in _ps._STATIC_FIELDS:
            kwargs[name] = getattr(result_pmap, name)
        all_packed[c] = _ps.PackedCrossCatState(**kwargs)

    # Log joint scores
    scores = [float(packed_log_joint(all_packed[c], full_data)) for c in range(N_CHAINS)]
    for c in range(N_CHAINS):
        all_metrics[c].append({"sweep": sweep, "log_joint": scores[c]})

    print(
        f"  Sweep {sweep:3d}/{N_POST_SWEEPS}: {elapsed:.1f}s, "
        f"log_joints=[{', '.join(f'{s:.0f}' for s in scores)}]"
    )

# Select best chain
best_scores = [float(packed_log_joint(p, full_data)) for p in all_packed]
best_chain = int(np.argmax(best_scores))
print(f"\nBest chain: {best_chain + 1} (log_joint={best_scores[best_chain]:.0f})")

## 7. Convergence Plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for c in range(N_CHAINS):
    sweeps = [m["sweep"] for m in all_metrics[c]]
    ljs = [m["log_joint"] for m in all_metrics[c]]
    ax.plot(sweeps, ljs, "o-", label=f"Chain {c + 1}")
ax.set_xlabel("Sweep")
ax.set_ylabel("Log Joint")
ax.set_title(f"100K Row Benchmark -- {N_CHAINS} chains on {n_devices} GPUs (pmap)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save Results

In [ ]:
from pathlib import Path

results_dir = Path("benchmarks/results/scaling")
results_dir.mkdir(parents=True, exist_ok=True)

results = {
    "backend": platform["backend"],
    "devices": [str(d) for d in jax.devices()],
    "n_devices": n_devices,
    "n_chains": N_CHAINS,
    "chains_per_device": CHAINS_PER_DEVICE,
    "n_rows": N_ROWS,
    "n_cols": N_COLS,
    "pre_sweep_time": pre_time,
    "insert_time": insert_time,
    "best_chain": best_chain,
    "best_log_joint": best_scores[best_chain],
    "per_sweep_metrics": all_metrics,
    "connector": {
        "save_time": save_time,
        "load_time": load_time,
        "file_size_mb": file_size_mb,
    },
}
with open(results_dir / "scaling_100k_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {results_dir / 'scaling_100k_results.json'}")

# Archive to /kaggle/working/ for easy download
import shutil

results_tar = "/kaggle/working/scaling_100k_results.tar.gz"
shutil.make_archive("/kaggle/working/scaling_100k_results", "gztar", ".", str(results_dir))
print(f"Archived to {results_tar}")
print("Download from Kaggle Output tab.")

for f in sorted(results_dir.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        print(f"  {f.relative_to(results_dir)}  ({size:,} bytes)")